# S3 J4 — MCP Client

Cette journée construit le pendant client du MCP Server du jour 3.

Objectif : découvrir les tools d'un serveur MCP, valider les arguments et exposer ces tools à un agent.

## 1. Rappel architectural

Un client MCP est une couche d'adaptation.

Il ne raisonne pas à la place de l'agent. Il rend les capacités serveur utilisables, testables, validables et traçables.

In [ ]:
from pathlib import Path
import sys

# Exécution depuis le notebook généré ou depuis le dossier du lab.
lab_path = Path("../../book/week03/day04/labs").resolve()
if lab_path.exists():
    sys.path.insert(0, str(lab_path))

In [ ]:
from mcp_client import build_demo_client

client = build_demo_client()
tools = client.list_tools()

for name, spec in tools.items():
    print(name, "dangerous=", spec.dangerous)

## 2. Appeler un tool

Le client valide les arguments puis appelle `tools/call`.

In [ ]:
result = client.call_tool("get_ticket", {"ticket_id": "INC-42"})
result

## 3. Adapter les tools MCP à l'agent

L'agent peut recevoir un dictionnaire de fonctions appelables.

In [ ]:
registry = client.as_agent_tools()
observation = registry["search_knowledge_base"]({"query": "reset MFA", "limit": 1})
observation.structured_content

## 4. Sécurité

Certains tools doivent être bloqués côté client sans approbation explicite.

In [ ]:
try:
    client.call_tool("refund_customer", {"customer_id": "CUST-42", "amount": 12.5})
except Exception as exc:
    print(type(exc).__name__, str(exc))

## 5. Trace

Une trace exploitable permet de debugger les appels MCP.

In [ ]:
print(client.trace_json())

## Exercices

1. Ajoute un tool non dangereux côté serveur simulé.
2. Ajoute un champ optionnel dans son schéma.
3. Vérifie que le client valide correctement les types.
4. Expose ce tool via `as_agent_tools()`.
5. Ajoute une trace d'erreur lorsque la validation échoue.

# Notes formateur

Les corrections suivantes reprennent les fichiers `corriges/` du jour.

# Corrigé — Exercices MCP Client

## Exercice 1 — Responsabilités

| Responsabilité | Couche |
|---|---|
| Choisir le prochain outil à appeler | Agent |
| Retourner la liste des tools exposés | MCP Server |
| Valider localement les arguments | MCP Client |
| Exécuter une recherche dans une base support | Tool métier |
| Transformer un tool MCP en fonction Python appelable | MCP Client |
| Bloquer un outil sensible sans approbation | MCP Client ou guardrail |
| Synthétiser la réponse finale à l'utilisateur | Agent |

## Exercice 2 — Schéma

Champs obligatoires :

- `customer_id`
- `priority`

Types :

- `customer_id`: string
- `priority`: string
- `include_history`: boolean

Propriétés supplémentaires :

- interdites, car `additionalProperties` vaut `false`.

Exemple valide :

```json
{
  "customer_id": "CUST-42",
  "priority": "high",
  "include_history": true
}
```

Exemple invalide :

```json
{
  "customer_id": "CUST-42",
  "priority": "high",
  "debug": true
}
```

La propriété `debug` n'est pas déclarée.

## Exercice 3 — Trace

```json
{
  "request_id": 7,
  "tool": "get_ticket",
  "arguments": {"ticket_id": "INC-42"},
  "status": "success",
  "duration_ms": 12,
  "error": null
}
```

## Exercice 4 — Adapter MCP vers agent

```python
def as_agent_tool(client, tool_name):
    def tool(arguments):
        return client.call_tool(tool_name, arguments)
    return tool
```

## Exercice 5 — Cache

Le cache doit être invalidable car le serveur peut évoluer.

Situations dangereuses :

1. un tool a été supprimé ou renommé ;
2. un schéma d'entrée a changé ;
3. une permission a été modifiée ;
4. un tool sensible a été ajouté ;
5. une version serveur différente est déployée.

## Exercice 6 — Sécurité

Contrôles possibles avant `refund_customer` :

- vérifier une approbation humaine ;
- contrôler un montant maximum ;
- vérifier le rôle utilisateur ;
- journaliser l'appel ;
- exiger un ticket associé ;
- bloquer l'appel si le contexte est incomplet.

# Corrigé — Questions d'entretien MCP Client

## Question 1

Un client MCP est la couche qui connecte l'orchestrateur ou l'agent à un serveur MCP. Il initialise la session, découvre les capabilities, expose les tools localement, valide les arguments, appelle les méthodes MCP et convertit les résultats en observations exploitables.

## Question 2

Sans couche cliente dédiée, l'agent devient couplé au protocole, au transport et aux formats d'erreur. Cela rend le système plus fragile, plus difficile à tester et plus difficile à sécuriser.

## Question 3

Une erreur locale de validation est détectée avant l'appel serveur, par exemple un argument obligatoire manquant. Une erreur serveur MCP vient de la réponse du serveur, par exemple un tool inconnu ou une erreur d'exécution.

## Question 4

Le client appelle `tools/list`, convertit chaque tool en spécification locale, puis crée une fonction wrapper par tool. Chaque wrapper appelle `client.call_tool(tool_name, arguments)`.

## Question 5

Le cache peut devenir dangereux si les tools, schémas, permissions ou politiques serveur changent. Il faut prévoir une invalidation manuelle, TTL ou invalidation sur erreur.

## Question 6

On instrumente le client avec des traces contenant request id, méthode, tool, arguments filtrés, statut, durée, erreur éventuelle et métadonnées de session.

## Question 7

Le client doit bloquer avant le serveur si les arguments sont invalides, si le tool n'est pas autorisé, si une action sensible manque d'approbation ou si une politique de sécurité échoue.

## Question 8

Une action sensible doit passer par une politique explicite : approbation humaine, rôle utilisateur, seuils, audit, traçabilité et éventuellement confirmation multi-étape.

# Corrigé — Challenge MCP Client

## Approche

Une solution robuste doit séparer :

1. le transport JSON-RPC ;
2. la découverte des tools ;
3. la validation ;
4. la politique de sécurité ;
5. l'adaptation en registry agentique.

## Exemple de flux

```python
transport = InMemoryTransport(InMemoryMCPServer())
client = MCPClient(transport)

client.initialize()
tools = client.list_tools()
result = client.call_tool("get_ticket", {"ticket_id": "INC-42"})
```

## Points de validation attendus

- `initialize` doit être appelé avant les tools.
- `tools/list` doit retourner des définitions exploitables.
- `tools/call` doit refuser les arguments invalides.
- `refund_customer` doit être bloqué sans `approved=True`.
- les traces doivent conserver les succès et les erreurs.

## Améliorations possibles

- Ajouter un TTL au cache.
- Masquer les champs sensibles dans les traces.
- Ajouter un budget d'appels par session.
- Supporter plusieurs serveurs MCP.
- Associer chaque tool à une politique d'autorisation.

# Review formateur — MCP Client

## Résumé pédagogique

Cette journée est le pendant client du jour 3.

Les apprenants doivent comprendre qu'un serveur MCP expose des capacités, mais que le client est la couche qui rend ces capacités utilisables par un agent en production.

## Points à vérifier

- L'apprenant distingue agent, client MCP, serveur MCP et tool métier.
- L'apprenant sait expliquer `initialize`, `tools/list` et `tools/call`.
- L'apprenant comprend pourquoi la validation locale est utile.
- L'apprenant sait construire une registry d'outils dynamique.
- L'apprenant sait expliquer les risques du cache.
- L'apprenant sait justifier le blocage d'une action sensible.

## Erreurs fréquentes

1. Confondre client MCP et agent.
2. Mettre la planification dans le client.
3. Ne pas valider les arguments.
4. Ne pas tracer les appels.
5. Considérer le cache comme une source de vérité permanente.
6. Laisser un tool sensible s'exécuter sans approbation.

## Questions de relance

- Que se passe-t-il si le serveur change son schéma ?
- Où placerais-tu une politique d'autorisation ?
- Comment testerais-tu ce client sans serveur réel ?
- Comment adapterais-tu ce client pour plusieurs serveurs MCP ?
- Comment masquerais-tu les secrets dans les traces ?

## Barème indicatif

| Critère | Points |
|---|---:|
| Séparation des responsabilités | 4 |
| Initialisation et découverte | 4 |
| Validation locale | 4 |
| Gestion d'erreurs | 3 |
| Sécurité | 3 |
| Traces | 2 |
| Total | 20 |